In [1]:
# Persipan
import numpy as np
import pandas as pd

np.random.seed(111)
n = 2500
menu_list = ["Kopi Susu", "Americano", "Cappuccino", "Roti Bakar", "Kentang Goreng", "Es Teh"]
cabang_list = ["Magelang", "Yogyakarta", "Semarang"]
waktu_list = ["Pagi", "Siang", "Sore", "Malam"]

data_tugas11 = pd.DataFrame({
    "nama_menu": np.random.choice(menu_list, size=n),
    "cabang": np.random.choice(cabang_list, size=n),
    "waktu_pesan": np.random.choice(waktu_list, size=n),
    "qty": np.random.randint(1, 6, size=n),
    "harga_satuan": np.random.choice([15000, 18000, 20000, 22000, 25000, 30000], size=n),
})
data_tugas11["total_penjualan"] = data_tugas11["qty"] * data_tugas11["harga_satuan"] + np.random.normal(0, 8000, size=n)
data_tugas11.to_csv("tugas11_penjualan.csv", index=False)

print(f"Dataset berhasil dibuat: {len(data_tugas11)} baris")
print(data_tugas11.head())

Dataset berhasil dibuat: 2500 baris
        nama_menu      cabang waktu_pesan  qty  harga_satuan  total_penjualan
0  Kentang Goreng    Semarang       Malam    1         30000     27750.163532
1  Kentang Goreng  Yogyakarta        Sore    3         20000     64216.160744
2  Kentang Goreng    Semarang       Siang    5         22000    108722.792592
3  Kentang Goreng  Yogyakarta        Pagi    2         20000     54483.205872
4      Roti Bakar    Magelang       Malam    2         30000     58157.944120


In [4]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

spark = SparkSession.builder \
    .appName("Tugas11-KopiNusantara") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
spark.conf.set("spark.sql.shuffle.partitions", "4")

df = spark.read.csv("tugas11_penjualan.csv", header=True, inferSchema=True)
df.show(5)
print("Total baris:", df.count())
df.printSchema()

+--------------+----------+-----------+---+------------+------------------+
|     nama_menu|    cabang|waktu_pesan|qty|harga_satuan|   total_penjualan|
+--------------+----------+-----------+---+------------+------------------+
|Kentang Goreng|  Semarang|      Malam|  1|       30000|27750.163531558544|
|Kentang Goreng|Yogyakarta|       Sore|  3|       20000| 64216.16074371864|
|Kentang Goreng|  Semarang|      Siang|  5|       22000|108722.79259154423|
|Kentang Goreng|Yogyakarta|       Pagi|  2|       20000|54483.205871792496|
|    Roti Bakar|  Magelang|      Malam|  2|       30000| 58157.94412015849|
+--------------+----------+-----------+---+------------+------------------+
only showing top 5 rows

Total baris: 2500
root
 |-- nama_menu: string (nullable = true)
 |-- cabang: string (nullable = true)
 |-- waktu_pesan: string (nullable = true)
 |-- qty: integer (nullable = true)
 |-- harga_satuan: integer (nullable = true)
 |-- total_penjualan: double (nullable = true)



In [5]:
# A.Feature Engineering
from pyspark.ml.feature import StringIndexer, VectorAssembler

# Simpan Model hasil fit()
indexer_menu = StringIndexer(inputCol="nama_menu", outputCol="nama_menu_idx").fit(df)
indexer_cabang = StringIndexer(inputCol="cabang", outputCol="cabang_idx").fit(df)
indexer_waktu = StringIndexer(inputCol="waktu_pesan", outputCol="waktu_pesan_idx").fit(df)

df_indexed = indexer_menu.transform(df)
df_indexed = indexer_cabang.transform(df_indexed)
df_indexed = indexer_waktu.transform(df_indexed)

df_indexed.select("nama_menu", "nama_menu_idx", "cabang", "cabang_idx",
                  "waktu_pesan", "waktu_pesan_idx").show(8)

assembler = VectorAssembler(
    inputCols=["qty", "harga_satuan", "nama_menu_idx", "cabang_idx", "waktu_pesan_idx"],
    outputCol="features"
)

df_final = assembler.transform(df_indexed)
df_final.select("features", "total_penjualan").show(5, truncate=False)

+--------------+-------------+----------+----------+-----------+---------------+
|     nama_menu|nama_menu_idx|    cabang|cabang_idx|waktu_pesan|waktu_pesan_idx|
+--------------+-------------+----------+----------+-----------+---------------+
|Kentang Goreng|          5.0|  Semarang|       2.0|      Malam|            1.0|
|Kentang Goreng|          5.0|Yogyakarta|       1.0|       Sore|            3.0|
|Kentang Goreng|          5.0|  Semarang|       2.0|      Siang|            2.0|
|Kentang Goreng|          5.0|Yogyakarta|       1.0|       Pagi|            0.0|
|    Roti Bakar|          4.0|  Magelang|       0.0|      Malam|            1.0|
|     Americano|          0.0|  Magelang|       0.0|       Pagi|            0.0|
|    Cappuccino|          2.0|  Magelang|       0.0|      Malam|            1.0|
|    Cappuccino|          2.0|  Semarang|       2.0|       Pagi|            0.0|
+--------------+-------------+----------+----------+-----------+---------------+
only showing top 8 rows

+--

In [6]:
#B. Train/Test Split & Pelatihan Model
from pyspark.ml.regression import LinearRegression

train, test = df_final.randomSplit([0.75, 0.25], seed=101)

print(f"Jumlah data training : {train.count()}")
print(f"Jumlah data testing  : {test.count()}")

lr = LinearRegression(featuresCol="features", labelCol="total_penjualan")
model = lr.fit(train)

print("Model berhasil dilatih!")
print("Koefisien tiap fitur:", model.coefficients)
print("Intercept:", model.intercept)

Jumlah data training : 1839
Jumlah data testing  : 661


Model berhasil dilatih!
Koefisien tiap fitur: [22046.663271025755,3.0229208681606696,-123.21425598657828,279.2845246167609,339.63879532317213]
Intercept: -66903.1776648588


In [7]:
# C. Evaluasi Model
from pyspark.ml.evaluation import RegressionEvaluator

predictions = model.transform(test)
predictions.select("nama_menu", "cabang", "waktu_pesan", "qty", "harga_satuan",
                   "total_penjualan", "prediction").show(10)

evaluator_rmse = RegressionEvaluator(labelCol="total_penjualan", predictionCol="prediction", metricName="rmse")
evaluator_r2 = RegressionEvaluator(labelCol="total_penjualan", predictionCol="prediction", metricName="r2")

rmse = evaluator_rmse.evaluate(predictions)
r2 = evaluator_r2.evaluate(predictions)

print(f"RMSE : Rp {rmse:,.0f}")
print(f"R²   : {r2:.4f}  ({r2*100:.1f}% variasi data berhasil dijelaskan model)")

+---------+--------+-----------+---+------------+------------------+------------------+
|nama_menu|  cabang|waktu_pesan|qty|harga_satuan|   total_penjualan|        prediction|
+---------+--------+-----------+---+------------+------------------+------------------+
|Americano|Magelang|      Malam|  1|       25000|19721.170446051627|31056.146105506865|
|Americano|Magelang|      Malam|  1|       25000|  21236.0956379149|31056.146105506865|
|Americano|Magelang|      Malam|  4|       22000|  85834.8564401191| 88127.37331410211|
|Americano|Magelang|      Malam|  4|       22000|102174.09100317833| 88127.37331410211|
|Americano|Magelang|      Malam|  4|       30000|113044.63431555482|112310.74025938749|
|Americano|Magelang|      Malam|  5|       22000|115810.29600214415|110174.03658512786|
|Americano|Magelang|      Malam|  5|       25000|140386.40273615732|119242.79918960987|
|Americano|Magelang|       Pagi|  2|       30000| 73842.64420790318|  67877.7749220128|
|Americano|Magelang|       Pagi|

In [8]:
# D. Eksperimen Perbandingan Fitur
# Model kedua: hanya qty dan harga_satuan, memakai train/test split yang SAMA (seed=101)
assembler_sederhana = VectorAssembler(
    inputCols=["qty", "harga_satuan"],
    outputCol="features_sederhana"
)

train_s = assembler_sederhana.transform(train)
test_s = assembler_sederhana.transform(test)

lr_sederhana = LinearRegression(featuresCol="features_sederhana", labelCol="total_penjualan")
model_sederhana = lr_sederhana.fit(train_s)
pred_sederhana = model_sederhana.transform(test_s)

r2_sederhana = RegressionEvaluator(labelCol="total_penjualan", predictionCol="prediction", metricName="r2").evaluate(pred_sederhana)
rmse_sederhana = RegressionEvaluator(labelCol="total_penjualan", predictionCol="prediction", metricName="rmse").evaluate(pred_sederhana)

print(f"Model utama (5 fitur)       -> R²: {r2:.4f} | RMSE: Rp {rmse:,.0f}")
print(f"Model kedua (qty + harga)   -> R²: {r2_sederhana:.4f} | RMSE: Rp {rmse_sederhana:,.0f}")
print(f"Selisih R² (kedua - utama)  : {r2_sederhana - r2:+.4f}")

Model utama (5 fitur)       -> R²: 0.9107 | RMSE: Rp 10,901
Model kedua (qty + harga)   -> R²: 0.9108 | RMSE: Rp 10,896
Selisih R² (kedua - utama)  : +0.0001


In [9]:
# E. Prediksi Skenario Baru
# 3 skenario dummy. Nilai kategori harus termasuk yang dikenal saat training.
skenario = spark.createDataFrame([
    ("Kopi Susu", "Magelang", "Pagi", 2, 18000),
    ("Roti Bakar", "Yogyakarta", "Sore", 4, 25000),
    ("Es Teh", "Semarang", "Malam", 5, 15000),
], ["nama_menu", "cabang", "waktu_pesan", "qty", "harga_satuan"])

# Pakai indexer MODEL yang SAMA dari bagian A (jangan membuat StringIndexer baru)
skenario_idx = indexer_menu.transform(skenario)
skenario_idx = indexer_cabang.transform(skenario_idx)
skenario_idx = indexer_waktu.transform(skenario_idx)

# Assembler yang sama dengan bagian A, lalu prediksi dengan model utama
skenario_final = assembler.transform(skenario_idx)
hasil = model.transform(skenario_final)

hasil.select("nama_menu", "cabang", "waktu_pesan", "qty", "harga_satuan", "prediction").show(truncate=False)

+----------+----------+-----------+---+------------+------------------+
|nama_menu |cabang    |waktu_pesan|qty|harga_satuan|prediction        |
+----------+----------+-----------+---+------------+------------------+
|Kopi Susu |Magelang  |Pagi       |2  |18000       |31479.510248098188|
|Roti Bakar|Yogyakarta|Sore       |4  |25000       |97661.84100990092 |
|Es Teh    |Semarang  |Malam      |5  |15000       |89202.51678927695 |
+----------+----------+-----------+---+------------+------------------+



In [18]:
# Tambahan
from pyspark.ml import Pipeline, PipelineModel
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler, StandardScaler
from pyspark.ml.regression import LinearRegression, RandomForestRegressor
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder

df_eks = spark.read.csv("tugas11_penjualan.csv", header=True, inferSchema=True)
train_eks, test_eks = df_eks.randomSplit([0.8, 0.2], seed=42)

ev_r2 = RegressionEvaluator(labelCol="total_penjualan", predictionCol="prediction", metricName="r2")
ev_rmse = RegressionEvaluator(labelCol="total_penjualan", predictionCol="prediction", metricName="rmse")
print("Setup OK. Training:", train_eks.count(), "| Testing:", test_eks.count())

Setup OK. Training: 2051 | Testing: 449


In [19]:
idx_menu_m = StringIndexer(inputCol="nama_menu", outputCol="nama_menu_idx").fit(df_eks)
idx_cabang_m = StringIndexer(inputCol="cabang", outputCol="cabang_idx").fit(df_eks)
idx_waktu_m = StringIndexer(inputCol="waktu_pesan", outputCol="waktu_pesan_idx").fit(df_eks)

df_idx = idx_waktu_m.transform(idx_cabang_m.transform(idx_menu_m.transform(df_eks)))

enc_m = OneHotEncoder(
    inputCols=["nama_menu_idx", "cabang_idx", "waktu_pesan_idx"],
    outputCols=["nama_menu_ohe", "cabang_ohe", "waktu_pesan_ohe"]
).fit(df_idx)

enc_m.transform(df_idx).select("nama_menu", "nama_menu_idx", "nama_menu_ohe").show(5, truncate=False)

+--------------+-------------+-------------+
|nama_menu     |nama_menu_idx|nama_menu_ohe|
+--------------+-------------+-------------+
|Kentang Goreng|5.0          |(5,[],[])    |
|Kentang Goreng|5.0          |(5,[],[])    |
|Kentang Goreng|5.0          |(5,[],[])    |
|Kentang Goreng|5.0          |(5,[],[])    |
|Roti Bakar    |4.0          |(5,[4],[1.0])|
+--------------+-------------+-------------+
only showing top 5 rows



In [20]:
idx_menu = StringIndexer(inputCol="nama_menu", outputCol="nama_menu_idx")
idx_cabang = StringIndexer(inputCol="cabang", outputCol="cabang_idx")
idx_waktu = StringIndexer(inputCol="waktu_pesan", outputCol="waktu_pesan_idx")
encoder_eks = OneHotEncoder(
    inputCols=["nama_menu_idx", "cabang_idx", "waktu_pesan_idx"],
    outputCols=["nama_menu_ohe", "cabang_ohe", "waktu_pesan_ohe"]
)
assembler_ohe = VectorAssembler(
    inputCols=["qty", "harga_satuan", "nama_menu_ohe", "cabang_ohe", "waktu_pesan_ohe"],
    outputCol="features"
)
lr_p = LinearRegression(featuresCol="features", labelCol="total_penjualan")

pipeline_eks = Pipeline(stages=[idx_menu, idx_cabang, idx_waktu, encoder_eks, assembler_ohe, lr_p])

pipeline_model_eks = pipeline_eks.fit(train_eks)
pred_eks = pipeline_model_eks.transform(test_eks)

pred_eks.select("nama_menu", "cabang", "waktu_pesan", "total_penjualan", "prediction").show(5)
print(f"R² : {ev_r2.evaluate(pred_eks):.4f}")

+---------+--------+-----------+------------------+------------------+
|nama_menu|  cabang|waktu_pesan|   total_penjualan|        prediction|
+---------+--------+-----------+------------------+------------------+
|Americano|Magelang|      Malam|16176.640411161712|22613.797725145778|
|Americano|Magelang|      Malam| 25086.08815754784|31613.250737400595|
|Americano|Magelang|      Malam| 33315.70021727038|  32667.2539600241|
|Americano|Magelang|      Malam| 52186.94536069484| 60719.61621941204|
|Americano|Magelang|      Malam|100265.57617366312| 104825.0700551815|
+---------+--------+-----------+------------------+------------------+
only showing top 5 rows

R² : 0.9201


In [21]:
asm_num = VectorAssembler(inputCols=["qty", "harga_satuan"], outputCol="fitur_mentah")
train_num = asm_num.transform(train_eks)
test_num = asm_num.transform(test_eks)

# fit hanya pada training agar tidak ada kebocoran data dari testing
scaler_m = StandardScaler(inputCol="fitur_mentah", outputCol="features",
                          withStd=True, withMean=False).fit(train_num)
train_sc = scaler_m.transform(train_num)
test_sc = scaler_m.transform(test_num)

m_sc = LinearRegression(featuresCol="features", labelCol="total_penjualan").fit(train_sc)
print("Koefisien (skala seragam):", m_sc.coefficients)
print(f"R² : {ev_r2.evaluate(m_sc.transform(test_sc)):.4f}")

Koefisien (skala seragam): [31093.962744574914,14668.483517550601]
R² : 0.9202


In [22]:
for nama, reg, en in [("OLS", 0.0, 0.0), ("Ridge", 0.1, 0.0),
                      ("Lasso", 0.1, 1.0), ("ElasticNet", 0.1, 0.5)]:
    m = LinearRegression(featuresCol="features", labelCol="total_penjualan",
                         regParam=reg, elasticNetParam=en).fit(train_sc)
    r2 = ev_r2.evaluate(m.transform(test_sc))
    print(f"{nama:12s} R²: {r2:.4f} | koefisien: {m.coefficients}")

OLS          R²: 0.9202 | koefisien: [31093.962744574914,14668.483517550601]
Ridge        R²: 0.9202 | koefisien: [31093.87737800307,14668.444273045572]
Lasso        R²: 0.9202 | koefisien: [31093.864187034622,14668.385102618615]
ElasticNet   R²: 0.9202 | koefisien: [31093.870759342193,14668.414735991466]


In [23]:
asm_rf = VectorAssembler(
    inputCols=["qty", "harga_satuan", "nama_menu_idx", "cabang_idx", "waktu_pesan_idx"],
    outputCol="features"
)
rf = RandomForestRegressor(featuresCol="features", labelCol="total_penjualan",
                           numTrees=20, maxDepth=5, seed=42)

pipeline_rf = Pipeline(stages=[idx_menu, idx_cabang, idx_waktu, asm_rf, rf])
model_rf = pipeline_rf.fit(train_eks)
pred_rf = model_rf.transform(test_eks)

print(f"RandomForest -> R²: {ev_r2.evaluate(pred_rf):.4f} | RMSE: Rp {ev_rmse.evaluate(pred_rf):,.0f}")

nama_fitur = ["qty", "harga_satuan", "nama_menu_idx", "cabang_idx", "waktu_pesan_idx"]
for nama, nilai in zip(nama_fitur, model_rf.stages[-1].featureImportances.toArray()):
    print(f"{nama:16s}: {nilai:.4f}")

RandomForest -> R²: 0.8739 | RMSE: Rp 12,988
qty             : 0.7733
harga_satuan    : 0.2001
nama_menu_idx   : 0.0124
cabang_idx      : 0.0054
waktu_pesan_idx : 0.0088


In [25]:
param_grid = ParamGridBuilder() \
    .addGrid(lr_p.regParam, [0.0, 0.1]) \
    .addGrid(lr_p.elasticNetParam, [0.0, 1.0]) \
    .build()

cv = CrossValidator(estimator=pipeline_eks, estimatorParamMaps=param_grid,
                    evaluator=ev_rmse, numFolds=2, seed=42)
cv_model = cv.fit(train_eks)

print("RMSE rata-rata tiap kombinasi:", [round(x) for x in cv_model.avgMetrics])
best_lr = cv_model.bestModel.stages[-1]
print("regParam terbaik       :", best_lr.getRegParam())
print("elasticNetParam terbaik:", best_lr.getElasticNetParam())
print(f"R² model terbaik pada testing: {ev_r2.evaluate(cv_model.transform(test_eks)):.4f}")

RMSE rata-rata tiap kombinasi: [10699, 10699, 10699, 10699]
regParam terbaik       : 0.1
elasticNetParam terbaik: 1.0
R² model terbaik pada testing: 0.9201


In [26]:
pipeline_model_eks.write().overwrite().save("model_penjualan_eks")

model_dimuat = PipelineModel.load("model_penjualan_eks")
pred_dimuat = model_dimuat.transform(test_eks)

pred_dimuat.select("nama_menu", "cabang", "total_penjualan", "prediction").show(5)
print(f"R² model yang dimuat: {ev_r2.evaluate(pred_dimuat):.4f}")

+---------+--------+------------------+------------------+
|nama_menu|  cabang|   total_penjualan|        prediction|
+---------+--------+------------------+------------------+
|Americano|Magelang|16176.640411161712|22613.797725145778|
|Americano|Magelang| 25086.08815754784|31613.250737400595|
|Americano|Magelang| 33315.70021727038|  32667.2539600241|
|Americano|Magelang| 52186.94536069484| 60719.61621941204|
|Americano|Magelang|100265.57617366312| 104825.0700551815|
+---------+--------+------------------+------------------+
only showing top 5 rows

R² model yang dimuat: 0.9201


In [27]:
spark.stop()
print("SparkSession ditutup.")

SparkSession ditutup.
